In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load all datasets
path = "../data/raw/"

orders = pd.read_csv(path + "olist_orders_dataset.csv")
customers = pd.read_csv(path + "olist_customers_dataset.csv")
order_items = pd.read_csv(path + "olist_order_items_dataset.csv")
payments = pd.read_csv(path + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(path + "olist_order_reviews_dataset.csv")
products = pd.read_csv(path + "olist_products_dataset.csv")
sellers = pd.read_csv(path + "olist_sellers_dataset.csv")
category_translation = pd.read_csv(path + "product_category_name_translation.csv")

# Quick inspection loop
datasets = {
    "orders": orders, "customers": customers, "order_items": order_items,
    "payments": payments, "reviews": reviews, "products": products,
    "sellers": sellers
}

for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f"\n{name.upper()} — shape: {df.shape}")
    if len(missing) > 0:
        print(missing)
    else:
        print("No missing values")


ORDERS — shape: (99441, 8)
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

CUSTOMERS — shape: (99441, 5)
No missing values

ORDER_ITEMS — shape: (112650, 7)
No missing values

PAYMENTS — shape: (103886, 5)
No missing values

REVIEWS — shape: (99224, 7)
review_comment_title      87656
review_comment_message    58247
dtype: int64

PRODUCTS — shape: (32951, 9)
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

SELLERS — shape: (3095, 4)
No missing values


In [ ]:
# 1. Convert all date columns in orders to actual datetime
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# 2. Create a flag for delivery status instead of dropping missing rows
orders['is_delivered'] = orders['order_delivered_customer_date'].notnull()

# 3. Calculate delivery delay (actual vs estimated) — only where delivered
orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days

# 4. Handle missing product category — fill with 'unknown' rather than dropping rows
products['product_category_name'] = products['product_category_name'].fillna('unknown')

# 5. Reviews: don't drop nulls in comment text — instead create a flag
reviews['has_comment'] = reviews['review_comment_message'].notnull()

# Sanity check
print(orders[['order_purchase_timestamp', 'is_delivered', 'delivery_delay_days']].head())
print(f"\n% orders delivered late: {(orders['delivery_delay_days'] > 0).mean() * 100:.2f}%")

In [7]:
order_items_full = order_items.merge(products, on='product_id', how='left')
order_items_full = order_items_full.merge(category_translation, on='product_category_name', how='left')
order_items_full.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools


In [8]:
order_items_full = order_items.merge(products, on='product_id', how='left')
order_items_full = order_items_full.merge(category_translation, on='product_category_name', how='left')
print(order_items_full.shape)
print(order_items_full.head())

(112650, 16)
                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   
3  00024acbcdf0a6daa1e931b038114c75              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1   

                         product_id                         seller_id  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d   
3  7634da152a4610f1595efa32f14722fc  9d7a1d34a5052409006425275ba1c2b4   
4  ac6c3623068f30de03045865e4e10089  df560393f3a51e74553ab94004ba5c87   

   shipping_limit_date   price  freight_value product_category_name  \
0  2017-09-19 09:45:35   58.90          13.29            cool_stuff   
1  2017-05-03 11:05:13  239.90          19.93              pet_shop   
2  2018-01

In [9]:
order_summary = order_items_full.groupby('order_id').agg(
    total_items=('order_item_id', 'count'),
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    main_category=('product_category_name_english', lambda x: x.mode()[0] if not x.mode().empty else 'unknown')
).reset_index()

print(order_summary.shape)
print(order_summary.head())


(98666, 5)
                           order_id  total_items  total_price  total_freight  \
0  00010242fe8c5a6d1ba2dd792cb16214            1        58.90          13.29   
1  00018f77f2f0320c557190d7a144bdd3            1       239.90          19.93   
2  000229ec398224ef6ca0657da4fc703e            1       199.00          17.87   
3  00024acbcdf0a6daa1e931b038114c75            1        12.99          12.79   
4  00042b26cf59d7ce69dfabb4e55b4fd9            1       199.90          18.14   

     main_category  
0       cool_stuff  
1         pet_shop  
2  furniture_decor  
3        perfumery  
4     garden_tools  


In [10]:
payment_summary = payments.groupby('order_id').agg(
    total_payment_value=('payment_value', 'sum'),
    payment_type=('payment_type', lambda x: x.mode()[0] if not x.mode().empty else 'unknown'),
    installments=('payment_installments', 'max')
).reset_index()

print(payment_summary.shape)
print(payment_summary.head())

(99440, 4)
                           order_id  total_payment_value payment_type  \
0  00010242fe8c5a6d1ba2dd792cb16214                72.19  credit_card   
1  00018f77f2f0320c557190d7a144bdd3               259.83  credit_card   
2  000229ec398224ef6ca0657da4fc703e               216.87  credit_card   
3  00024acbcdf0a6daa1e931b038114c75                25.78  credit_card   
4  00042b26cf59d7ce69dfabb4e55b4fd9               218.04  credit_card   

   installments  
0             2  
1             3  
2             5  
3             2  
4             3  


In [11]:
review_summary = reviews.groupby('order_id').agg(
    review_score=('review_score', 'mean'),
    has_comment=('has_comment', 'max')
).reset_index()

print(review_summary.shape)
print(review_summary.head())

(98673, 3)
                           order_id  review_score  has_comment
0  00010242fe8c5a6d1ba2dd792cb16214           5.0         True
1  00018f77f2f0320c557190d7a144bdd3           4.0        False
2  000229ec398224ef6ca0657da4fc703e           5.0         True
3  00024acbcdf0a6daa1e931b038114c75           4.0        False
4  00042b26cf59d7ce69dfabb4e55b4fd9           5.0         True


In [12]:
master = orders.merge(customers, on='customer_id', how='left')
master = master.merge(order_summary, on='order_id', how='left')
master = master.merge(payment_summary, on='order_id', how='left')
master = master.merge(review_summary, on='order_id', how='left')

print(master.shape)
print(master.head())

(99441, 23)
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp   order_approved_at  \
0    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06 2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39 2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2

In [19]:
master.to_csv("../data/processed/master_ecommerce.csv", index=False)
print("Saved!")

Saved!


In [20]:
import os
os.makedirs("../data/processed", exist_ok=True)

master.to_csv("../data/processed/master_ecommerce.csv", index=False)
print("Saved!")

Saved!
